# 02_analysis.ipynb - Analitik Distribusi FMCG

Notebook ini menjalankan 5 metrik analitik utama:
1. **Ghost Outlet Detection** - Outlet 90 hari tanpa transaksi
2. **Route Compliance** - Planned vs actual visits
3. **Visit Duration Summary** - Statistik durasi kunjungan
4. **Sales Performance** - Sales + churn risk flag
5. **Prospect Potential** - Weighted score untuk outlet baru

**ATURAN:** Notebook ini idempotent - bisa dijalankan ulang dari awal.

In [ ]:
# ── SETUP ────────────────────────────────────────────────────────────────
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

import pandas as pd
from config import DATA_DIR, OUT_DIR, GHOST_DAYS, CHURN_THRESHOLD
from analysis import (
    detect_ghost_outlets,
    calc_route_compliance,
    visit_duration_summary,
    sales_performance,
    calc_prospect_potential
)

print("✅ Setup complete")

In [ ]:
# ── LOAD DATA DARI PARQUET ──────────────────────────────────────────────
print("📂 Loading data from Parquet...")

df_doccall = pd.read_parquet(DATA_DIR / "doccall.parquet")
df_sales = pd.read_parquet(DATA_DIR / "sales.parquet")
df_customer = pd.read_parquet(DATA_DIR / "customer.parquet")
df_address = pd.read_parquet(DATA_DIR / "address.parquet")
df_employee = pd.read_parquet(DATA_DIR / "employee.parquet")

print(f"  ✅ doccall: {len(df_doccall):,} rows")
print(f"  ✅ sales: {len(df_sales):,} rows")
print(f"  ✅ customer: {len(df_customer):,} rows")
print(f"  ✅ address: {len(df_address):,} rows")
print(f"  ✅ employee: {len(df_employee):,} rows")

In [ ]:
# ── METRIK 1: GHOST OUTLET DETECTION ────────────────────────────────────
# Patch 1: Deteksi outlet tidak aktif > 90 hari

print("\n" + "="*60)
print("👻 METRIK 1: GHOST OUTLET DETECTION")
print("="*60)

ghost_outlets = detect_ghost_outlets(
    df_sales=df_sales,
    df_customer=df_customer,
    ghost_days=GHOST_DAYS
)

print(f"\nTotal outlet: {len(df_customer):,}")
print(f"Ghost outlet (> {GHOST_DAYS} hari): {len(ghost_outlets):,}")
print(f"Ghost rate: {len(ghost_outlets)/len(df_customer)*100:.1f}%")

if len(ghost_outlets) > 0:
    print("\nTop 10 Ghost Outlets by Branch:")
    print(ghost_outlets.groupby('szBranchId').size().sort_values(ascending=False).head(10))

In [ ]:
# ── METRIK 2: ROUTE COMPLIANCE ──────────────────────────────────────────
# Patch 1: Planned vs actual visits

print("\n" + "="*60)
print("🛣️ METRIK 2: ROUTE COMPLIANCE")
print("="*60)

# Note: Untuk route compliance yang akurat, perlu data planned route dari Excel
# Di sini kita gunakan pendekatan sederhana: frekuensi kunjungan per outlet

route_compliance = calc_route_compliance(
    df_doccall=df_doccall,
    df_customer=df_customer
)

print(f"\nOutlet dengan kunjungan:")
print(f"  Regular (≥2x/bulan): {len(route_compliance[route_compliance['visits_per_month'] >= 2]):,}")
print(f"  Irregular (<2x/bulan): {len(route_compliance[route_compliance['visits_per_month'] < 2]):,}")

print("\nTop 10 Route dengan Kunjungan Terbanyak:")
top_routes = route_compliance.groupby('szRouteId').agg({
    'szCustomerId': 'count',
    'visits_per_month': 'mean'
}).rename(columns={'szCustomerId': 'total_visits'}).sort_values('total_visits', ascending=False).head(10)
print(top_routes)

In [ ]:
# ── METRIK 3: VISIT DURATION SUMMARY ────────────────────────────────────
# Rule #5: decDuration dalam DETIK (bukan menit!)

print("\n" + "="*60)
print("⏱️ METRIK 3: VISIT DURATION SUMMARY")
print("="*60)

duration_stats = visit_duration_summary(df_doccall=df_doccall)

print("\nStatistik Durasi Kunjungan (dalam detik):")
print(f"  Mean: {duration_stats['mean_duration_sec']:.0f} detik ({duration_stats['mean_duration_sec']/60:.1f} menit)")
print(f"  Median: {duration_stats['median_duration_sec']:.0f} detik")
print(f"  Std Dev: {duration_stats['std_duration_sec']:.0f} detik")
print(f"  Min: {duration_stats['min_duration_sec']:.0f} detik")
print(f"  Max: {duration_stats['max_duration_sec']:.0f} detik")

print("\nDurasi per Branch (top 10):")
print(duration_stats['by_branch'].head(10))

In [ ]:
# ── METRIK 4: SALES PERFORMANCE & CHURN RISK ────────────────────────────
# Patch 3: Churn risk flag jika sales < 70% bulan lalu

print("\n" + "="*60)
print("💰 METRIK 4: SALES PERFORMANCE & CHURN RISK")
print("="*60)

sales_perf = sales_performance(
    df_sales=df_sales,
    df_customer=df_customer,
    churn_threshold=CHURN_THRESHOLD
)

print(f"\nTotal outlet dengan transaksi: {len(sales_perf):,}")
print(f"Churn risk (< {CHURN_THRESHOLD*100:.0f}%): {len(sales_perf[sales_perf['churn_risk'] == True]):,}")
print(f"Churn rate: {len(sales_perf[sales_perf['churn_risk'] == True])/len(sales_perf)*100:.1f}%")

print("\nTop 10 Outlet dengan Penurunan Sales Terbesar:")
churn_sorted = sales_perf[sales_perf['churn_risk'] == True].sort_values('sales_change_pct').head(10)
print(churn_sorted[['szCustomerId', 'current_sales', 'previous_sales', 'sales_change_pct']])

In [ ]:
# ── METRIK 5: PROSPECT POTENTIAL (Scoring) ──────────────────────────────
# Patch 3: Weighted scoring untuk outlet baru

print("\n" + "="*60)
print("🎯 METRIK 5: PROSPECT POTENTIAL SCORING")
print("="*60)

# Gabungkan data customer + address untuk scoring
df_prospect = df_customer.merge(df_address[['szCustomerId', 'szLatitude', 'szLongitude', 'szCity']], 
                                 on='szCustomerId', how='left')

# Merge dengan sales data (untuk yang sudah ada transaksi)
prospect_scored = calc_prospect_potential(
    df_customer=df_prospect,
    df_sales=df_sales
)

print(f"\nTotal outlet discoring: {len(prospect_scored):,}")
print(f"\nDistribusi Score:")
print(prospect_scored['prospect_score'].describe())

print("\nTop 10 Outlet dengan Score Tertinggi:")
print(prospect_scored.sort_values('prospect_score', ascending=False).head(10)[
    ['szCustomerId', 'szCustomerName', 'prospect_score', 'has_sales']
])

In [ ]:
# ── SIMPAN HASIL ANALISIS ───────────────────────────────────────────────
print("\n" + "="*60)
print("💾 SAVING RESULTS")
print("="*60)

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Simpan semua hasil analisis
ghost_outlets.to_csv(OUT_DIR / "ghost_outlets.csv", index=False)
print(f"  ✅ ghost_outlets.csv ({len(ghost_outlets):,} rows)")

route_compliance.to_csv(OUT_DIR / "route_compliance.csv", index=False)
print(f"  ✅ route_compliance.csv ({len(route_compliance):,} rows)")

sales_perf.to_csv(OUT_DIR / "sales_performance.csv", index=False)
print(f"  ✅ sales_performance.csv ({len(sales_perf):,} rows)")

prospect_scored.to_csv(OUT_DIR / "prospect_score.csv", index=False)
print(f"  ✅ prospect_score.csv ({len(prospect_scored):,} rows)")

print("\n✅ Semua hasil analisis tersimpan!")

In [ ]:
# ── RINGKASAN ───────────────────────────────────────────────────────────
print("\n" + "="*60)
print("✅ ANALISIS SELESAI")
print("="*60)
print(f"\n📊 Key Metrics:")
print(f"   👻 Ghost outlets: {len(ghost_outlets):,} ({len(ghost_outlets)/len(df_customer)*100:.1f}%)")
print(f"   ⚠️  Churn risk: {len(sales_perf[sales_perf['churn_risk'] == True]):,} ({len(sales_perf[sales_perf['churn_risk'] == True])/len(sales_perf)*100:.1f}%)")
print(f"   ⏱️  Avg visit duration: {duration_stats['mean_duration_sec']/60:.1f} menit")
print(f"   🎯 High potential prospects: {len(prospect_scored[prospect_scored['prospect_score'] >= 70]):,}")
print("\n🚀 Lanjut ke notebook 03_map_visualization.ipynb")